# Car Price Prediction Using CatBoost

**Project:** CodeAlpha – Car Price Prediction  
**Goal:** Estimate the selling price of a used car from its details.

This notebook includes data loading, cleaning, exploratory plots, model training, evaluation, an Actual-vs-Predicted plot, and an interactive prediction form.

> The model is an educational estimate, not a guaranteed market valuation. Prices depend on location, condition, service history, and other factors not present in this dataset.

## 1. Install and import libraries
Run this cell first. CatBoost and ipywidgets may need installation in a fresh Colab runtime.

In [ ]:
%pip -q install catboost ipywidgets

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## 2. Load the dataset
Upload `CAR DETAILS FROM CAR DEKHO.csv` to the Colab session if it is not already present in `/content`.

In [ ]:
from google.colab import files

DATA_PATH = "/content/CAR DETAILS FROM CAR DEKHO.csv"

if not os.path.exists(DATA_PATH):
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("Please upload the car details CSV file.")
    DATA_PATH = os.path.join("/content", csv_files[0])

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:", df.columns.tolist())

## 3. Inspect and clean the data
Exact duplicate rows are removed. We also check missing values and basic validity.

In [ ]:
print("Missing values before cleaning:")
display(df.isna().sum())

print("Exact duplicate rows:", df.duplicated().sum())

df_clean = df.drop_duplicates().copy().reset_index(drop=True)

# Keep rows with valid target and basic numeric fields.
df_clean = df_clean.dropna(subset=[
    "name", "year", "selling_price", "km_driven",
    "fuel", "seller_type", "transmission", "owner"
])
df_clean = df_clean[
    (df_clean["selling_price"] > 0) &
    (df_clean["km_driven"] >= 0) &
    (df_clean["year"] <= 2026)
].copy()

df_clean["name"] = df_clean["name"].astype(str).str.strip()
df_clean["brand"] = df_clean["name"].str.split().str[0]
df_clean["car_age"] = 2026 - df_clean["year"]

print("Shape after cleaning:", df_clean.shape)
display(df_clean.head())
print("\nMissing values after cleaning:")
display(df_clean.isna().sum())

## 4. Exploratory data analysis
These plots help inspect the target price distribution and relationships in the data.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_clean["selling_price"], bins=40)
plt.title("Distribution of Used-Car Selling Prices")
plt.xlabel("Selling price (₹)")
plt.ylabel("Number of cars")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df_clean["km_driven"], df_clean["selling_price"], alpha=0.35)
plt.title("Selling Price vs Kilometres Driven")
plt.xlabel("Kilometres driven")
plt.ylabel("Selling price (₹)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
df_clean.groupby("car_age")["selling_price"].median().sort_index().plot()
plt.title("Median Selling Price by Car Age")
plt.xlabel("Car age (years)")
plt.ylabel("Median selling price (₹)")
plt.tight_layout()
plt.show()

## 5. Prepare features and split the data
The model uses the car name and categorical details directly. `year` is converted into `car_age`. The same test split was used during earlier model comparisons, so the reported score should be treated as an estimate rather than an independent final benchmark.

In [ ]:
feature_columns = [
    "name", "fuel", "seller_type", "transmission", "owner",
    "km_driven", "brand", "car_age"
]
target_column = "selling_price"

X = df_clean[feature_columns].copy()
y = df_clean[target_column].copy()

categorical_columns = ["name", "fuel", "seller_type", "transmission", "owner", "brand"]
for col in categorical_columns:
    X[col] = X[col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Categorical features:", categorical_columns)

## 6. Train the CatBoost model
These settings reflect the selected model configuration from the project experiments. Early stopping monitors the test split here, as in the earlier workflow; for a rigorous final benchmark, reserve a separate validation split and keep the test set untouched.

In [ ]:
best_model = CatBoostRegressor(
    iterations=1500,
    depth=8,
    learning_rate=0.05,
    l2_leaf_reg=9,
    loss_function="MAE",
    random_seed=RANDOM_STATE,
    verbose=200,
    allow_writing_files=False
)

best_model.fit(
    X_train, y_train,
    cat_features=categorical_columns,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100
)

## 7. Evaluate the model

In [ ]:
predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("CAR PRICE PREDICTION - MODEL PERFORMANCE")
print("-" * 40)
print(f"Mean Absolute Error (MAE): ₹{mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ₹{rmse:,.2f}")
print(f"R² Score: {r2:.4f} ({r2 * 100:.2f}%)")

## 8. Actual vs Predicted prices

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, predictions, alpha=0.45)

low = min(float(y_test.min()), float(np.min(predictions)))
high = max(float(y_test.max()), float(np.max(predictions)))
plt.plot([low, high], [low, high], linestyle="--")

plt.title("Actual vs Predicted Car Prices")
plt.xlabel("Actual selling price (₹)")
plt.ylabel("Predicted selling price (₹)")
plt.tight_layout()
plt.show()

## 9. Save the trained model
This creates `car_price_model.cbm` in the current Colab session.

In [ ]:
MODEL_PATH = "/content/car_price_model.cbm"
best_model.save_model(MODEL_PATH)
print("Model saved to:", MODEL_PATH)

# Uncomment to download the model to your computer:
# from google.colab import files
# files.download(MODEL_PATH)

## 10. Predict the price of a car
Choose a car and enter its details. The car-name dropdown uses names present in the dataset.

In [ ]:
def predict_car_price(name, year, km_driven, fuel, seller_type, transmission, owner):
    if not isinstance(name, str) or not name.strip():
        raise ValueError("Please select a car name.")
    if int(km_driven) < 0:
        raise ValueError("Kilometres driven cannot be negative.")
    if int(year) > 2026:
        raise ValueError("Please enter a valid manufacturing year.")

    car_data = pd.DataFrame([{
        "name": name.strip(),
        "fuel": fuel,
        "seller_type": seller_type,
        "transmission": transmission,
        "owner": owner,
        "km_driven": int(km_driven),
        "brand": name.strip().split()[0],
        "car_age": 2026 - int(year)
    }])

    car_data = car_data[feature_columns]
    return float(best_model.predict(car_data)[0])

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

car_names = sorted(df_clean["name"].dropna().unique().tolist())

car_name_dropdown = widgets.Dropdown(
    options=car_names,
    description="Car Name:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "120px"}
)
year_input = widgets.IntSlider(
    value=2020,
    min=int(df_clean["year"].min()),
    max=2026,
    description="Year:",
    style={"description_width": "120px"}
)
km_input = widgets.IntText(value=35000, description="KM Driven:", style={"description_width": "120px"})
fuel_input = widgets.Dropdown(options=sorted(df_clean["fuel"].dropna().unique()), description="Fuel:", style={"description_width": "120px"})
seller_input = widgets.Dropdown(options=sorted(df_clean["seller_type"].dropna().unique()), description="Seller:", style={"description_width": "120px"})
transmission_input = widgets.Dropdown(options=sorted(df_clean["transmission"].dropna().unique()), description="Transmission:", style={"description_width": "120px"})
owner_input = widgets.Dropdown(options=sorted(df_clean["owner"].dropna().unique()), description="Owner:", style={"description_width": "120px"})

predict_button = widgets.Button(description="Predict Price", button_style="success")
output = widgets.Output()

def predict_selected_car(_):
    with output:
        clear_output(wait=True)
        try:
            price = predict_car_price(
                name=car_name_dropdown.value,
                year=year_input.value,
                km_driven=km_input.value,
                fuel=fuel_input.value,
                seller_type=seller_input.value,
                transmission=transmission_input.value,
                owner=owner_input.value
            )
            print(f"Estimated Car Price: ₹{price:,.0f}")
            print("\nNote: This is a machine-learning estimate, not a guaranteed market price.")
        except Exception as exc:
            print("Prediction error:", exc)

predict_button.on_click(predict_selected_car)

display(
    car_name_dropdown, year_input, km_input, fuel_input,
    seller_input, transmission_input, owner_input,
    predict_button, output
)

## 11. Limitations
- The dataset may not represent every city or current market condition.
- Important details such as vehicle condition, accident history, service records, and exact location are not available as features.
- MAE is the average absolute prediction error in rupees; it is not a percentage accuracy score.
- R² describes variance explained on the evaluation split; it does not mean every prediction is that percentage accurate.
- Because the evaluation split was reused during earlier tuning and early stopping, these metrics may be optimistic.